### Finetune three models

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('******.pkl')
df_final

In [ ]:
df_final.columns

In [ ]:
features = [
    'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
]

In [ ]:
import numpy as np
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Category distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from sklearn.base import clone   
from sklearn.model_selection import GridSearchCV   

# Data Preparation
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

#Split out an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} Subject, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} Subject, {len(y_test)} records")
print(f"Proportion of positive cases in Test Set: {y_test.mean():.1%}")

#K×K function to find the optimal configuration for nested CV
def find_optimal_kxk(pipeline_template, param_grid, model_name, X_tv, y_tv, groups_tv):
    print(f"\n=== {model_name} Beginning to iterate through 81 nested CV configurations, ranging from 2×2 to 10×10 (using full GridSearchCV) ===")
    
    best_score = -np.inf
    best_outer, best_inner = None, None
    
    for outer_k in range(2, 11):
        for inner_k in range(2, 11):
            print(f"  test {outer_k}×{inner_k} ... ", end="")
            
            outer_gkf = GroupKFold(n_splits=outer_k)
            fold_aucs = []
            
            for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
                X_train = X_tv[train_idx]
                y_train = y_tv[train_idx]
                g_train = groups_tv[train_idx]
                
                #Use a full GridSearchCV (iterating through all parameters)
                search = GridSearchCV(
                    estimator=clone(pipeline_template),
                    param_grid=param_grid,           
                    scoring='roc_auc',
                    cv=GroupKFold(n_splits=inner_k),
                    n_jobs=-1,
                    verbose=0
                )
                
                search.fit(X_train, y_train, groups=g_train)
                proba_val = search.best_estimator_.predict_proba(X_tv[val_idx])[:, 1]
                fold_aucs.append(roc_auc_score(y_tv[val_idx], proba_val))
            
            avg_auc = np.mean(fold_aucs)
            print(f"Avg AUC: {avg_auc:.4f}")
            
            if avg_auc > best_score:
                best_score = avg_auc
                best_outer, best_inner = outer_k, inner_k
    
    print(f"\n✅ {model_name} Optimal Configuration: {best_outer}×{best_inner} (Avg AUC = {best_score:.4f})")
    return best_outer, best_inner

#Define three models
rf_pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(class_weight='balanced', random_state=42))])
param_grid_rf = {'rf__n_estimators': [50,100,200], 'rf__max_depth': [None,10,20], 'rf__min_samples_split': [2,5,10]}

svm_pipeline = Pipeline([('scaler', StandardScaler()), ('svm', SVC(class_weight='balanced', probability=True, random_state=42))])
param_grid_svm = {'svm__C': [0.1,1,10], 'svm__kernel': ['linear','rbf','poly'], 'svm__degree': [2,3,4]}

cb_pipeline = Pipeline([('cb', CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=0))])
param_grid_cb = {'cb__iterations': [100,200,300], 'cb__learning_rate': [0.01,0.1,0.2],
                 'cb__depth': [4,6,8], 'cb__l2_leaf_reg': [1,3,5]}

#Iterate to find the optimal K×K configuration
rf_best_outer, rf_best_inner = find_optimal_kxk(rf_pipeline, param_grid_rf, "Random Forest", X_tv, y_tv, groups_tv)
svm_best_outer, svm_best_inner = find_optimal_kxk(svm_pipeline, param_grid_svm, "SVM", X_tv, y_tv, groups_tv)
cb_best_outer, cb_best_inner = find_optimal_kxk(cb_pipeline, param_grid_cb, "CatBoost", X_tv, y_tv, groups_tv)

#Train the final model using the optimal configuration + evaluate on the test set
def train_final_model_and_evaluate(pipeline_template, param_grid, model_name,
                                   outer_k, inner_k,
                                   X_tv, y_tv, groups_tv,
                                   X_test, y_test):
    
    final_search = GridSearchCV(
        estimator=clone(pipeline_template),
        param_grid=param_grid,
        scoring='roc_auc',
        cv=GroupKFold(n_splits=inner_k),
        n_jobs=-1,
        verbose=0
    )
    final_search.fit(X_tv, y_tv, groups=groups_tv)
    
    final_model = final_search.best_estimator_
    final_params = final_search.best_params_
    print(f"{model_name} Final optimal parameters: {final_params}")
    
    proba_test = final_model.predict_proba(X_test)[:, 1]
    pred_test = (proba_test >= 0.5).astype(int)
    
    print(f"\n=== {model_name} Final performance on an independent test set ===")
    print(f"AUC: {roc_auc_score(y_test, proba_test):.4f}")
    print(f"Accuracy: {accuracy_score(y_test, pred_test):.1%}")
    print(f"F1-Score: {f1_score(y_test, pred_test):.4f}")
    print(f"Precision: {precision_score(y_test, pred_test):.4f}")
    print(f"Recall: {recall_score(y_test, pred_test):.4f}")
    
    tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
    print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")
    
    return final_model, final_params

#Run the final evaluation
print("\n=== Final training and test set evaluation using the optimal K×K configuration ===")

rf_final, rf_params = train_final_model_and_evaluate(rf_pipeline, param_grid_rf, "Random Forest",
                                                     rf_best_outer, rf_best_inner, X_tv, y_tv, groups_tv, X_test, y_test)

svm_final, svm_params = train_final_model_and_evaluate(svm_pipeline, param_grid_svm, "SVM",
                                                       svm_best_outer, svm_best_inner, X_tv, y_tv, groups_tv, X_test, y_test)

cb_final, cb_params = train_final_model_and_evaluate(cb_pipeline, param_grid_cb, "CatBoost",
                                                     cb_best_outer, cb_best_inner, X_tv, y_tv, groups_tv, X_test, y_test)

### Re-train three models

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('********.pkl')
df_final

In [ ]:
df_final.columns

In [ ]:
features = [
    'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'
]

In [ ]:
import numpy as np
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Category distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

#Data preparation
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

#Split out an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} Subject, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} Subject, {len(y_test)} records")
print(f"Positive Proportion in Test Set: {y_test.mean():.1%}")

#Build a pipeline using fixed optimal parameters
rf_base = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42
)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

#Perform OOF (Out-of-Fold) analysis on the Train+Validation set to find the optimal threshold.

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr


# Set the minimum sensitivity you need
min_sensitivity = 0.70         

#Filter out all thresholds that satisfy Sensitivity >= min_sensitivity
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower min_sensitivity.")
    optimal_threshold = 0.5
else:
    #Among the thresholds that meet the criteria, select the one with the highest specificity.
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target Minimum Sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponds to Sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

#Train the final model on the complete Train+Validation set
print("\n=== Train the final model on the complete Train+Validation set ===")
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

#Make predictions on an independent test set using the optimal threshold.
print("\n=== Final performance on the independent test set (using the optimal threshold) ===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

#Data preparation
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

#Split out an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Positive Proportion in Test Set: {y_test.mean():.1%}")

#Build a pipeline using fixed optimal parameters
#Final optimal parameters for SVM: {'svm__C': 0.1, 'svm__degree': 2, 'svm__kernel': 'rbf'}
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

#Perform OOF (Out-of-Fold) analysis on the Train+Validation set to find the optimal threshold.
print("\n===Calculate OOF probabilities on the Train+Validation set and find the optimal threshold. ===")

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Set the minimum sensitivity you need
min_sensitivity = 0.70          

#Filter out all thresholds that satisfy Sensitivity >= min_sensitivity
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower min_sensitivity.")
    optimal_threshold = 0.5
else:
    #Among the thresholds that meet the criteria, select the one with the highest specificity.
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target Minimum Sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponds to Sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

#Train the final model on the complete Train+Validation set
print("\n=== Train the final model on the complete Train+Validation set ===")
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Make predictions on an independent test set using the optimal threshold.
print("\n=== Final performance on the independent test set (using the optimal threshold)===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)  


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
# CatBoostClassifier import
from catboost import CatBoostClassifier

# Data preparation
X = df_final[features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split out an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Positive Proportion in Test Set: {y_test.mean():.1%}")

#Build a pipeline using fixed optimal parameters
rf_base =CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=0, depth=8, iterations=300, l2_leaf_reg=1, learning_rate=0.01)

rf_pipeline = Pipeline([
    ('rf', rf_base)
])

#Perform OOF (Out-of-Fold) analysis on the Train+Validation set to find the optimal threshold.
print("\n=== Calculate OOF probabilities on the Train+Validation set and find the optimal threshold. ===")

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

#Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

#Set the minimum sensitivity you need
min_sensitivity = 0.70          

#Filter out all thresholds that satisfy Sensitivity >= min_sensitivity
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower min_sensitivity.")
    optimal_threshold = 0.5
else:
    #Among the thresholds that meet the criteria, select the one with the highest specificity.
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target Minimum Sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponds to Sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete Train+Validation set
print("\n=== Train the final model on the complete Train+Validation set ===")
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

#Make predictions on an independent test set using the optimal threshold.
print("\n=== Final performance on the independent test set (using the optimal threshold) ===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)  

auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")